# Milestone 1 – Blockchain Ledger
## Smart Logistics Tracking System
**MO-IT148 | Blockchain and IoT Integration**

This notebook:
1. Connects to a local Ganache blockchain via Web3.py
2. Loads the deployed `IoTDataStorage` smart contract
3. Stores all 100 IoT logistics records on-chain
4. Verifies data retrieval

**Prerequisites:**
- Ganache running at `http://127.0.0.1:7545`
- `IoTDataStorage.sol` deployed via Remix IDE
- `logistics_iot_data.csv` in the same folder

---
## SECTION 1: SETTING UP THE ENVIRONMENT
### Connect to Ganache blockchain

In [1]:
from web3 import Web3

# Connect to local Ganache blockchain
ganache_url = 'http://127.0.0.1:7545'
web3 = Web3(Web3.HTTPProvider(ganache_url))

print('========================================')
print('  SETTING UP THE ENVIRONMENT')
print('========================================')

if web3.is_connected():
    print('✅ Connected to Ganache successfully!')
    print(f'   Was your connection successful? YES')
    print(f'   Chain ID  : {web3.eth.chain_id}')
    print(f'   Block #   : {web3.eth.block_number}')
    print(f'   Accounts  : {len(web3.eth.accounts)} available')
else:
    print('❌ Connection failed. Ensure Ganache is running.')
    print('   Was your connection successful? NO')

  SETTING UP THE ENVIRONMENT
✅ Connected to Ganache successfully!
   Was your connection successful? YES
   Chain ID  : 1337
   Block #   : 308
   Accounts  : 10 available


---
## SECTION 2: SMART CONTRACT INTERACTION
### Load the deployed IoTDataStorage contract

In [2]:
# ── REPLACE WITH YOUR REMIX DEPLOYED CONTRACT ADDRESS ─────────────────
contract_address = '0x769776d52E0987DAeF5B97D91CbA226A105e4c18'

abi = [
    {
        'inputs': [],
        'name': 'getTotalRecords',
        'outputs': [{'internalType': 'uint256', 'name': '', 'type': 'uint256'}],
        'stateMutability': 'view',
        'type': 'function'
    },
    {
        'inputs': [{'internalType': 'uint256', 'name': 'index', 'type': 'uint256'}],
        'name': 'getRecord',
        'outputs': [
            {'internalType': 'uint256', 'name': '', 'type': 'uint256'},
            {'internalType': 'uint256', 'name': '', 'type': 'uint256'}, 
            {'internalType': 'string',  'name': '', 'type': 'string'},
            {'internalType': 'string',  'name': '', 'type': 'string'},
            {'internalType': 'string',  'name': '', 'type': 'string'}
        ],
        'stateMutability': 'view',
        'type': 'function'
    },
    {
        'inputs': [
            {'internalType': 'uint256', 'name': '_dataTimestamp', 'type': 'uint256'},
            {'internalType': 'string',  'name': '_deviceId',  'type': 'string'},
            {'internalType': 'string',  'name': '_dataType',  'type': 'string'},
            {'internalType': 'string',  'name': '_dataValue', 'type': 'string'}
        ],
        'name': 'storeData',
        'outputs': [],
        'stateMutability': 'nonpayable',
        'type': 'function'
    }
]
# ──────────────────────────────────────────────────────────────────────

# Load the smart contract
contract_address = Web3.to_checksum_address(contract_address)
contract = web3.eth.contract(address=contract_address, abi=abi)

# Set the default sender address (first account from Ganache)
web3.eth.default_account = web3.eth.accounts[0]

# Check total records BEFORE storing
total_before = contract.functions.getTotalRecords().call()

print('========================================')
print('  SMART CONTRACT INTERACTION')
print('========================================')
print(f'✅ Connected to Smart Contract at {contract_address}')
print(f'   Default Sender Account         : {web3.eth.default_account}')
print(f'   Total Records Before Storing   : {total_before}')
print(f'   Errors while loading contract? : NO')

  SMART CONTRACT INTERACTION
✅ Connected to Smart Contract at 0x769776d52E0987DAeF5B97D91CbA226A105e4c18
   Default Sender Account         : 0x65FC6AB2ad61F147d3F3a257E369CaF552e3a4eD
   Total Records Before Storing   : 0
   Errors while loading contract? : NO


---
## SECTION 3: STORING IoT DATA ON THE BLOCKCHAIN
### Part 1 – Load IoT sensor data from CSV (Homework 1)

In [3]:
import pandas as pd

# Load IoT sensor data from CSV (Generated in Homework 1)
df = pd.read_csv('logistics_iot_data.csv')

print('========================================')
print('  STORING IoT DATA — CSV PREVIEW')
print('========================================')
print(f'   Number of records in CSV file  : {len(df)}')
print(f'   Sensor types (data_type)       : {df["data_type"].unique().tolist()}')
print(f'   Records per sensor type        : {df["data_type"].value_counts().to_dict()}')
print()
print('   First 3 records in CSV:')
print(df[['timestamp','device_id','data_type','data_value','status']].head(3).to_string(index=False))

  STORING IoT DATA — CSV PREVIEW
   Number of records in CSV file  : 100
   Sensor types (data_type)       : ['GPS', 'Temperature', 'Humidity']
   Records per sensor type        : {'Temperature': 34, 'GPS': 33, 'Humidity': 33}

   First 3 records in CSV:
          timestamp device_id data_type           data_value     status
2026-01-01 01:30:10    RFID_0       GPS 14.949141,124.334215    Delayed
2026-01-01 02:30:22    RFID_1       GPS 10.528973,124.910757  Delivered
2026-01-01 03:28:12    RFID_2       GPS 11.412766,124.552706 In Transit


### Part 2 – Store each IoT record as a blockchain transaction

In [4]:
import time

def send_iot_data(data_timestamp, device_id, data_type, data_value):
    """Sends IoT data to the deployed smart contract.

    Args:
        data_timestamp : unix timestamp of the original sensor reading
        device_id      : RFID tag (e.g. RFID_0)
        data_type      : Sensor type — Temperature | Humidity | GPS
        data_value     : Sensor reading (e.g. 6.24C, 72.5%, lat,lon)
    """
    txn = contract.functions.storeData(
        data_timestamp, device_id, data_type, data_value
    ).transact({
        'from': web3.eth.default_account,
        'gas': 3000000
    })
    receipt = web3.eth.wait_for_transaction_receipt(txn)
    print(f'✅ Data Stored: {data_type} - {data_value}, Txn Hash: {receipt.transactionHash.hex()}')
    return receipt

print('========================================')
print('  STORING IoT DATA — TRANSACTIONS')
print('========================================')
print(f'Sending {len(df)} records to blockchain...')
print()

first_txn_hash = None
last_txn_hash  = None
success_count  = 0

# Make sure timestamp column is datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

for i, (_, row) in enumerate(df.iterrows()):
    data_ts = int(row['timestamp'].timestamp())  # convert to unix timestamp
    receipt = send_iot_data(
        data_ts,
        str(row['device_id']),
        str(row['data_type']),
        str(row['data_value'])
    )
    if i == 0:
        first_txn_hash = receipt.transactionHash.hex()
    last_txn_hash = receipt.transactionHash.hex()
    success_count += 1
    time.sleep(1)

print()
print('========================================')
print('  TRANSACTION SUMMARY')
print('========================================')
print(f'   Records successfully stored        : {success_count}')
print(f'   Transaction Hash of FIRST record   : {first_txn_hash}')
print(f'   Transaction Hash of LAST  record   : {last_txn_hash}')

  STORING IoT DATA — TRANSACTIONS
Sending 100 records to blockchain...

✅ Data Stored: GPS - 14.949141,124.334215, Txn Hash: 4b45e74bac36e3fc7d0816c7ccd5d2519c2fd7be977caaebd291dec4a5dcf2b3
✅ Data Stored: GPS - 10.528973,124.910757, Txn Hash: 4af55f99d66b0e6a8351f161578bb48261af4bfb75ee5a575ab2373c29119251
✅ Data Stored: GPS - 11.412766,124.552706, Txn Hash: 8a02c8eca6a777873cb08c72d9cdb688a98c574e5b617565267509875f2efe5e
✅ Data Stored: Temperature - 5.15C, Txn Hash: bd72c808bdadffbce8a4b553d2229ac81264c712cce1abcc24a0f3a36dd78a50
✅ Data Stored: Temperature - 7.31C, Txn Hash: 8ed3f8baa19cc48650ad4c3fe53b32d19ca0fc7efbdf5857812a3b531885dae3
✅ Data Stored: Temperature - 4.53C, Txn Hash: 586fef1a48501316abaf6752c85530fa0d297e1b7d29c36477d3f9cbccf8f7ba
✅ Data Stored: Temperature - 5.53C, Txn Hash: 7206ad67f0a1762da59e6683923bf18641ff7bf45680b69de4035e71d940eea0
✅ Data Stored: Temperature - 7.19C, Txn Hash: bef43131dfc15b6c71a22b9e1fa456642722f1cf9ad0048dbdc06538b8f925e9
✅ Data Stored: GPS 

---
## SECTION 4: RETRIEVING & VERIFYING DATA

In [5]:
import datetime

total_records = contract.functions.getTotalRecords().call()
record = contract.functions.getRecord(0).call()

latest_block = web3.eth.get_block('latest')
block_ts = datetime.datetime.fromtimestamp(
    latest_block['timestamp'], datetime.timezone.utc
).strftime('%Y-%m-%d %H:%M:%S')

# record[0] = dataTimestamp, record[1] = blockTimestamp, record[2] = deviceId, ...
sensor_ts = datetime.datetime.fromtimestamp(record[0], datetime.timezone.utc).strftime('%Y-%m-%d %H:%M:%S')

print('========================================')
print('  RETRIEVING & VERIFYING DATA')
print('========================================')
print(f'   Total IoT records stored: {total_records}')
print()
print('First Stored Record:', record)
print()
print('   First stored record retrieved from blockchain:')
print(f'   Sensor Timestamp  : {sensor_ts} UTC')
print(f'   Block Timestamp   : {block_ts} UTC')
print(f'   Device ID   : {record[2]}')
print(f'   Data Type   : {record[3]}')
print(f'   Data Value  : {record[4]}')

  RETRIEVING & VERIFYING DATA
   Total IoT records stored: 100

First Stored Record: [1767231010, 1783269420, 'RFID_0', 'GPS', '14.949141,124.334215']

   First stored record retrieved from blockchain:
   Sensor Timestamp  : 2026-01-01 01:30:10 UTC
   Block Timestamp   : 2026-07-05 16:38:46 UTC
   Device ID   : RFID_0
   Data Type   : GPS
   Data Value  : 14.949141,124.334215
